In [ ]:
#EDA part
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data=pd.read_excel("/content/World_development_mesurement.xlsx")
data.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/World_development_mesurement.xlsx'

In [ ]:
# Assuming your DataFrame is called 'data'
unique_countries = data['Country'].nunique()
print("Number of unique countries:", unique_countries)

In [ ]:
data.shape

In [ ]:
data.isnull().sum()

In [ ]:
data.info()

In [ ]:
# Cleaning money related columns because datatypes are reading abnormally
money_cols = ["GDP", "Health Exp/Capita", "Tourism Inbound", "Tourism Outbound"]

def clean_money(col):
    return (
        col.astype(str)                # converting to string
           .str.replace("$", "")       # removing $
           .str.replace(",", "")       # removing commas
           .str.strip()                # cleaning spaces
           .replace(["", "nan", "NaN"], None)  # handling missing
    )

# Applying cleaning and converting to numeric
for col in money_cols:
    data[col] = clean_money(data[col])
    data[col] = pd.to_numeric(data[col], errors="coerce")

# Cleaning Business Tax Rate
data["Business Tax Rate"] = (
    data["Business Tax Rate"]
    .astype(str)
    .str.replace("%", "")
    .str.strip()
)
data["Business Tax Rate"] = pd.to_numeric(data["Business Tax Rate"], errors="coerce")

# Checking conversion
print(data.dtypes)

In [ ]:
# Dropping columns that are not important for clusterig
columns_to_drop = [
    "Ease of Business",      # 93% missing
    "Business Tax Rate",     # too many nulls, weak indicator
    "Hours to do Tax",       # nearly half missing
    "Days to Start Business",# incomplete data
    "Number of Records"      # metadata
]

data = data.drop(columns=columns_to_drop)

data.info()

In [ ]:
data.isnull().sum()

In [ ]:
data.shape

In [ ]:
# replacing Null values in numeric columns using median
numeric_cols = data.select_dtypes(include=["float64", "int64"]).columns

for col in numeric_cols:
    median_value = data[col].median()
    data[col].fillna(median_value, inplace=True)

In [ ]:
data.isnull().sum()

In [ ]:
data.describe()

In [ ]:
#Plotting numerical columns to see how they distributed
num_cols = data.select_dtypes(include=['float64', 'int64']).columns

print("Plotting histograms with KDE for numeric columns...\n")

# for loop for all numerical columns
for col in num_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(data[col], kde=True, bins=30)
    plt.title(f"Distribution of {col}", fontsize=14)
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

In [ ]:
# Bivariate Analysis
# Comparing two variables at a time to understand their relationships.
# Scatterplots help us check trends and possible correlations.
# Each plot compares two variables to reveal trends.

sns.set(style="whitegrid")

# GDP vs Health Expenditure
plt.figure(figsize=(8,5))
sns.scatterplot(data=data, x="GDP", y="Health Exp/Capita")
plt.title("GDP vs Health Expenditure per Capita")
plt.xlabel("GDP")
plt.ylabel("Health Expenditure per Capita")
plt.show()

# Energy Usage vs CO2 Emissions
plt.figure(figsize=(8,5))
sns.scatterplot(data=data, x="Energy Usage", y="CO2 Emissions")
plt.title("Energy Usage vs CO2 Emissions")
plt.xlabel("Energy Usage")
plt.ylabel("CO2 Emissions")
plt.show()

# Birth Rate vs Infant Mortality
plt.figure(figsize=(8,5))
sns.scatterplot(data=data, x="Birth Rate", y="Infant Mortality Rate")
plt.title("Birth Rate vs Infant Mortality Rate")
plt.xlabel("Birth Rate")
plt.ylabel("Infant Mortality Rate")
plt.show()

# Internet Usage vs Life Expectancy
plt.figure(figsize=(8,5))
sns.scatterplot(data=data, x="Internet Usage", y="Life Expectancy Female")
plt.title("Internet Usage vs Life Expectancy (Female)")
plt.xlabel("Internet Usage")
plt.ylabel("Life Expectancy (Female)")
plt.show()

# GDP vs Internet Usage
plt.figure(figsize=(8,5))
sns.scatterplot(data=data, x="GDP", y="Internet Usage")
plt.title("GDP vs Internet Usage")
plt.xlabel("GDP")
plt.ylabel("Internet Usage")
plt.show()

In [ ]:
# boxplots to identify outliers
numeric_cols = data.select_dtypes(include=['float64', 'int64']).columns

plt.figure(figsize=(15, 40))

# for loop for all numerical columns
for i, col in enumerate(numeric_cols, 1):
    plt.subplot(len(numeric_cols), 1, i)
    sns.boxplot(x=data[col])
    plt.title(f"Boxplot of {col}")
    plt.xlabel("")

plt.tight_layout()
plt.show()


In [ ]:
# Detecting outliers using the IQR method
iqr_summary = {}

for col in numeric_cols:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Count how many values fall outside the bounds
    outliers = data[(data[col] < lower_bound) | (data[col] > upper_bound)][col].count()

    iqr_summary[col] = {
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": outliers
    }

# Converting into DataFrame for better understanding
iqr_summary_df = pd.DataFrame(iqr_summary).T
print(iqr_summary_df)

In [ ]:
# Correlation Heatmap
# Select only numeric columns
numeric_cols = data.select_dtypes(include=['float64', 'int64']).columns

corr_matrix = data[numeric_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# frequency of countires that are repeated
plt.figure(figsize=(10,6))
top10 = data["Country"].value_counts().head(10)

sns.barplot(x=top10.values, y=top10.index)
plt.title("Top 10 Most Frequent Countries in the Dataset")
plt.xlabel("Count")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

In [ ]:
# Saving cleaned dataset
data.to_csv("cleaned_Global_clustering.csv", index=False)